<h1>Model_Predict<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Import-libraries-&amp;-File" data-toc-modified-id="Import-libraries-&amp;-File-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Import libraries &amp; File</a></span></li><li><span><a href="#Preprocessing-functions" data-toc-modified-id="Preprocessing-functions-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Preprocessing functions</a></span></li><li><span><a href="#Load-the-model-and-print-out-the-score" data-toc-modified-id="Load-the-model-and-print-out-the-score-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Load the model and print out the score</a></span></li></ul></div>

### Import libraries & File

In [1]:
import warnings
warnings.filterwarnings('ignore')

# Data manip
import pandas as pd
import numpy as np

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.metrics import r2_score,max_error,mean_absolute_error, mean_squared_error

#Pickle for testing
import pickle

In [17]:
# File opening
data = pd.read_csv("data_init.txt") #Put your path file containing the datas to test
df = data.copy()
df=df.drop(["Unnamed: 0"], axis=1, errors='ignore')
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-119.84,36.77,6.0,1853.0,473.0,1397.0,417.0,1.4817,72000.0,INLAND
1,-117.80,33.68,8.0,2032.0,349.0,862.0,340.0,6.9133,274100.0,<1H OCEAN
2,-120.19,36.60,25.0,875.0,214.0,931.0,214.0,1.5536,58300.0,INLAND
3,-118.32,34.10,31.0,622.0,229.0,597.0,227.0,1.5284,200000.0,<1H OCEAN
4,-121.23,37.79,21.0,1922.0,373.0,1130.0,372.0,4.0815,117900.0,INLAND


### Preprocessing functions

In [3]:
# imputation function
def imputation (trainset,testset):
    trainset = trainset.dropna(axis=0)
    testset = testset.dropna(axis=0)
    return trainset,testset

# One-Hot encoding function
def ohe(trainset,testset):
    
    # Apply one-hot encoder to each column with categorical data
    OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
    OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(trainset.select_dtypes("object")))
    OH_cols_test = pd.DataFrame(OH_encoder.transform(testset.select_dtypes("object")))

    # One-hot encoding removed index; put it back
    OH_cols_train.index = trainset.index
    OH_cols_test.index = testset.index

    # Remove categorical columns (will replace with one-hot encoding)
    num_trainset = trainset.drop(trainset.select_dtypes("object"), axis=1)
    num_testset = testset.drop(testset.select_dtypes("object"), axis=1)

    # Add one-hot encoded columns to numerical features
    OH_trainset = pd.concat([num_trainset, OH_cols_train], axis=1)
    OH_testset = pd.concat([num_testset, OH_cols_test], axis=1)
    return OH_trainset, OH_trainset

# logarithm scaler function
def log_scale(trainset,testset):
    trainset[['total_rooms',"total_bedrooms", "population", "households"]] = np.log(trainset[['total_rooms',"total_bedrooms", "population", "households"]])
    testset[['total_rooms',"total_bedrooms", "population", "households"]] = np.log(trainset[['total_rooms',"total_bedrooms", "population", "households"]])
    return trainset,testset

# Standard scaler function
def standard(trainset, testset):
    ss = StandardScaler()
    trainset[['total_rooms',"total_bedrooms", "population", "households", "median_income","housing_median_age"]] = ss.fit_transform(trainset[['total_rooms',"total_bedrooms", "population", "households","median_income","housing_median_age"]])
    testset[['total_rooms',"total_bedrooms", "population", "households", "median_income","housing_median_age"]] = ss.transform(testset[['total_rooms',"total_bedrooms", "population", "households","median_income","housing_median_age"]])
    return trainset, testset

In [4]:
# Preprocessing function calling all functions above

def preprocessing(trainset, testset):
    trainset, testset = imputation(trainset, testset)
    trainset, testset = log_scale(trainset,testset)
    trainset, testset = standard(trainset, testset)
    trainset, testset = ohe(trainset, testset)
    
    X_train=trainset.drop("median_house_value", axis=1)
    y_train=trainset["median_house_value"]

    X_test=trainset.drop("median_house_value", axis=1)
    y_test=trainset["median_house_value"]
    
    return X_train, y_train, X_test, y_test

In [15]:
# evaluation model function

def evaluation(model):
    ypred = model.predict(X_test)
    
    print("R²:", r2_score(y_test, ypred))
    print("ME:", max_error(y_test, ypred))
    print("MAE:", mean_absolute_error(y_test,ypred))
    print("MSE:", mean_squared_error(y_test, ypred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, ypred)))
    print("prediction:", ypred)

In [6]:
trainset, testset = train_test_split(df, test_size=0.3, random_state=0)
X_train, y_train, X_test, y_test = preprocessing(trainset, testset)

### Load the model and print out the score

In [7]:
model = KNeighborsRegressor(n_neighbors = 2)

In [16]:
my_model = pickle.load(open("model.pkl","rb"))
evaluation(my_model)

R²: 0.907644106632221
ME: 216850.5
MAE: 22871.648420407808
MSE: 1233108163.3948762
RMSE: 35115.63986879459
prediction: [122650.  342750.  224850.  ... 213750.  147000.  397600.5]
